In [1]:
import numpy as np
import pandas as pd
import os
import datetime as dt
import time
from copy import deepcopy
import matplotlib.pyplot as plt

from f1_elo.pvp.general import build_pvp_results
from f1_elo.pvp_model.dataset import get_model_ready_pvp_results
from f1_elo.season import calc_season_drivers

import warnings
warnings.filterwarnings("ignore")

In [2]:
from f1_elo.pvp.fetch import get_results_data
from f1_elo.pvp.calc import (
    calc_pvp_results,
    filter_results,
)

from f1_elo.fetch_pvp import read_season_drivers

In [3]:
build_pvp_results()
calc_season_drivers()

In [4]:
results = get_results_data()
results['tournament'] = 'F1'
results = filter_results(results=results)
# results = results.rename(columns={'date': 'game_date', 'year': 'season'})
results['season'] = results['season'].astype(str)

In [5]:
season_drivers = read_season_drivers()
season_drivers = season_drivers.rename(columns={'team': 'driver'})
results = results.merge(season_drivers, how='inner', on=['tournament', 'season', 'driver'])
results = results.sort_values(['season', 'round', 'position'], ascending=True)
results['position'] = 1
results['position'] = results.groupby(['season', 'round'])['position'].cumsum()

In [6]:
results.tail(60)

,resultId,season,round,game_date,circuit,circuit_country,raceId,positionOrder,position,driver,constructor,points,grid,tournament
15056,26689,2024,21,2024-11-03,Autódromo José Carlos Pace,Brazil,1141,5,5,Charles Leclerc,Ferrari,10.0,6,F1
15057,26690,2024,21,2024-11-03,Autódromo José Carlos Pace,Brazil,1141,6,6,Lando Norris,McLaren,8.0,1,F1
15058,26691,2024,21,2024-11-03,Autódromo José Carlos Pace,Brazil,1141,7,7,Yuki Tsunoda,RB F1 Team,6.0,3,F1
15059,26692,2024,21,2024-11-03,Autódromo José Carlos Pace,Brazil,1141,8,8,Oscar Piastri,McLaren,4.0,8,F1
15060,26693,2024,21,2024-11-03,Autódromo José Carlos Pace,Brazil,1141,9,9,Liam Lawson,RB F1 Team,2.0,5,F1
15061,26694,2024,21,2024-11-03,Autódromo José Carlos Pace,Brazil,1141,10,10,Lewis Hamilton,Mercedes,1.0,14,F1
15062,26695,2024,21,2024-11-03,Autódromo José Carlos Pace,Brazil,1141,11,11,Sergio Pérez,Red Bull,0.0,12,F1
15063,26696,2024,21,2024-11-03,Autódromo José Carlos Pace,Brazil,1141,12,12,Oliver Bearman,Haas F1 Team,0.0,15,F1
15064,26697,2024,21,2024-11-03,Autódromo José Carlos Pace,Brazil,1141,13,13,Valtteri Bottas,Sauber,0.0,11,F1
15065,26698,2024,21,2024-11-03,Autódromo José Carlos Pace,Brazil,1141,14,14,Fernando Alonso,Aston Martin,0.0,9,F1


In [7]:
from copy import deepcopy
from datetime import date

from numpy import log as np_log
from pandas import (
    DataFrame,
    concat,
)

from f1_elo.constants import RESULT_RATIOS
from f1_elo.fetch_pvp import read_season_drivers


class EnduranceEloCalculation:
    """
    Class represents elo rating calculation logic.
    """
    def __init__(
        self,
        elo_game_value=20,
        new_agent_alpha=1.68,
        calibrated_rating=2000,
        num_calibrated_drivers=20,
        default_rating=1830.9,        
        saturation_rounds=10,
    ):
        """
        Initializes an instance with paths to data files and Elo configuration parameters.

        Arguments:
            elo_game_value (float): default Elo adjustment factor.
            new_agent_alpha (float): coefficient controlling the new agent adjustment.
            calibrated_rating (float): target mean rating for top drivers.
            num_calibrated_drivers (int): number of top drivers to use for calibration.
            default_rating (float): default Elo rating for new drivers.
            saturation_rounds (int): rounds required to reach rating stability.
        """
        self.elo_game_value = elo_game_value
        self.calibrated_rating = calibrated_rating
        self.num_calibrated_drivers = num_calibrated_drivers
        self.default_rating = default_rating
        self.saturation_rounds = saturation_rounds
        self.new_agent_alpha = new_agent_alpha

        self.season_drivers = read_season_drivers()
        self.drivers_years_active = self.season_drivers.groupby(['team']).agg(
            first_season=('season', 'min'),
            last_season=('season', 'max'),
        ).reset_index()

        self.elo_ratings_dict = {}
        self.num_games_per_team = {}

        self.temp_ratings = []
        self.total_rating = None
        self.output = []

    @property
    def log_loss(self):
        """
        Compute the mean log loss across all processed games.

        Returns:
            mean log loss if available as a float, otherwise None.
        """
        if len(self.output) == 0:
            return None
            
        output_df = pd.concat(self.output)
        log_loss = output_df['log_loss'].mean()

        return log_loss

    def reset_log_loss(self):
        """
        Reset stored log loss output.
        """
        self.output = []

    def run_pipeline(self, results):
        """
        Run ratings for every season bases on race results.

        Arguments:
            results (pandas.DataFrame): dataset with PVP interactions history.
        """
        seasons = sorted(results['season'].unique())

        for season in seasons:
            include_ix = self.drivers_years_active['first_season'] == season
            teams_to_include = self.drivers_years_active['team'].loc[include_ix].to_list()
            for team in teams_to_include:
                self.elo_ratings_dict[team] = self.elo_ratings_dict.get(team, self.default_rating)

            self._calibrate_ratings(season=season)

            self.run_season(results=results, season=season)
            exclude_ix = (self.drivers_years_active['last_season'] == season) & (season != '2024')
            teams_to_exclude = self.drivers_years_active['team'].loc[exclude_ix].to_list()
            for team in teams_to_exclude:
                _ = self.elo_ratings_dict.pop(team)

        self.total_rating = concat(self.temp_ratings)

    def run_round(self, round_results):
        """
        Run Elo updates for a single race.

        Arguments:
            round_results (pandas.DataFrame): Race results in PVP format.
        """
        # need a copy as expected result should be calculated on a pre race ratings for all race interactions.
        round_elo_rating_dict = deepcopy(self.elo_ratings_dict)
        round_df = round_results.copy(deep=True)

        round_df['elo_rating'] = round_df['driver'].map(self.elo_ratings_dict)
        round_df['weight'] = round_df['elo_rating'].map(lambda x: 10**(-x/400))
        round_df['num_rounds'] = round_df['driver'].map(self.num_games_per_team).fillna(0)
        round_df['new_agent_multiplier'] = round_df['num_rounds'].map(self._calc_new_agent_multiplier)

        while len(round_df) >= 2:
            ee_df = round_df[['driver', 'position', 'weight', 'new_agent_multiplier']]
            ee_df['probability_to_lose'] = ee_df['weight'] / ee_df['weight'].sum()
            num_losers = len(ee_df.loc[ee_df['position'] == ee_df['position'].max()])
            ee_df['is_last'] = ee_df['position'].map(lambda x: 1/num_losers if x == ee_df['position'].max() else 0)
            ee_df['elo_update'] = self.elo_game_value * ee_df['new_agent_multiplier'] * (ee_df['probability_to_lose'] - ee_df['is_last'])
            self.ee_df = ee_df
            
            elo_updates = ee_df.set_index('driver')['elo_update'].to_dict()
            for driver in elo_updates:
                round_elo_rating_dict[driver] = round_elo_rating_dict[driver] + elo_updates[driver]

            self.elo_updates = elo_updates

            round_df = round_df.loc[round_df['position'] != round_df['position'].max()]

        pvp_round_df = calc_pvp_results(round_results)
        pvp_round_df['result_ratio'] = pvp_round_df['result'].map(RESULT_RATIOS)
        pvp_round_df['exp_result'] = pvp_round_df.apply(self.calc_expected_result, axis=1)
        pvp_round_df['log_loss'] = pvp_round_df.apply(self._calc_log_loss, axis=1)
        self.output.append(pvp_round_df)
            
        self.elo_ratings_dict = round_elo_rating_dict

        round_drivers = set(round_results['driver'])
        for driver in round_drivers:
            self.num_games_per_team[driver] = self.num_games_per_team.get(driver, 0) + 1

    def run_season(self, results, season):
        """
        Execute Elo updates for all rounds within a season.

        Arguments:
            results (pandas.DataFrame): Dataset with race history.
            season (str): Season identifier.
        """
        self._log_current_rating(rating_date=date(int(season), 1, 1))
        season_rounds = sorted(results['round'].loc[results['season'] == season].unique())
        for season_round in season_rounds:
            round_results = results.loc[(results['season'] == season) & (results['round'] == season_round)]
            self.run_round(round_results=round_results)
            self._log_current_rating(rating_date=round_results['game_date'].iloc[0])

        self._log_current_rating(rating_date=date(int(season), 12, 31))

    @staticmethod
    def _calc_log_loss(interaction):
        """
        Calculates the log loss value for PVP interaction between two drivers.

        Arguments:
            interaction (dict): PVP interaction between two drivers.

        Returns:
            The elog loss value as a float.
        """
        res_ratio = interaction['result_ratio']
        exp_result = interaction['exp_result']
        log_loss = -res_ratio * np_log(exp_result) - (1 - res_ratio) * np_log(1 - exp_result)

        return log_loss

    def _calc_new_agent_multiplier(self, num_rounds):
        """
        Compute adjustment multiplier for newly introduced drivers.

        Arguments:
            num_rounds (int): Number of rounds a driver has participated in.

        Returns:
            Adjustment multiplier for rating change as a float.
        """
        multiplier = 1 + self.new_agent_alpha * (1 - min(self.saturation_rounds, num_rounds)/self.saturation_rounds)**2

        return multiplier

    def _calibrate_ratings(self, season):
        """
        Adjust all ratings so that the top drivers’ mean matches the calibrated rating.

        Arguments:
            season (str): Season identifier.
        """
        season_teams = self.season_drivers[['team']].loc[self.season_drivers['season'] == season]
        season_teams['rating'] = season_teams['team'].map(self.elo_ratings_dict)
        season_teams = season_teams.sort_values('rating', ascending=False).reset_index(drop=True)
        if len(season_teams) > self.num_calibrated_drivers:
            season_teams = season_teams.iloc[:self.num_calibrated_drivers]

        calibration_delta = self.calibrated_rating - season_teams['rating'].mean()
        for team in self.elo_ratings_dict:
            self.elo_ratings_dict[team] += calibration_delta

    def _log_current_rating(self, rating_date):
        """
        Log current Elo ratings with a specified date.

        Arguments:
            rating_date (datetime.date): Date of logged ratings.
        """
        temp_rating = DataFrame.from_dict(self.elo_ratings_dict, orient='index').reset_index()
        temp_rating = temp_rating.rename(columns={'index': 'team', 0: 'rating'})
        temp_rating['date'] = rating_date
        self.temp_ratings.append(temp_rating)

    def calc_expected_result(self, row):
        elo1 = self.elo_ratings_dict.get(row['home_team'])
        
        elo2 = self.elo_ratings_dict.get(row['away_team'])
        elo_diff = elo1 - elo2

        exp_res = 1 / (10 ** (-elo_diff / 400) + 1)

        return exp_res

In [8]:
obj = EnduranceEloCalculation(elo_game_value=45)

In [9]:
obj.run_pipeline(results=results)

In [10]:
obj.log_loss

0.47834585712183936

In [11]:
from f1_elo.constants import (
    INIT_DATASET_END_DATE,
    TRAIN_DATASET_END_DATE,
    VALIDATION_DATASET_END_DATE,
)
from f1_elo.gradient import AbstractGradientOptimizer

from numpy.random import uniform


class EnduranceEloGradientOptimizer(AbstractGradientOptimizer):
    """Gradient descent optimizer for tuning Elo model parameters."""

    def __init__(self, gradient_delta=0.0001):
        """
        Construct the object.

        Arguments:
            gradient_delta (float): Step size used for gradient perturbation.
        """
        super().__init__(gradient_delta=gradient_delta)
        self.init_settings = None

    def set_model(self):
        """
        Set the Elo model class to be optimized.
        """
        self.model = EnduranceEloCalculation

    def initialize_model_settings(self):
        """
        Initialize Elo model parameters with random starting values.

        Returns:
            Dictionary with initial Elo parameter settings.
        """
        settings = {'elo_game_value': uniform(10, 500), 'default_rating': uniform(1500, 2000),
                    'new_agent_alpha': uniform(0.1, 10)}

        self.init_settings = settings
        print(settings)

        return settings


def split_dataset(df):
    """
    Split a dataset into initialization, training, and validation subsets based on dates.

    Arguments:
        df (pandas.DataFrame): PVP results dataset.

    Returns:
        dict: Dictionary with keys 'init', 'train', and 'validation' containing the respective subsets.
    """
    init_ix = df['game_date'] <= INIT_DATASET_END_DATE
    init_df = df.loc[init_ix]

    train_ix = (df['game_date'] > INIT_DATASET_END_DATE) & (df['game_date'] <= TRAIN_DATASET_END_DATE)
    train_df = df.loc[train_ix]

    validation_ix = (df['game_date'] > TRAIN_DATASET_END_DATE) & (df['game_date'] <= VALIDATION_DATASET_END_DATE)
    validation_df = df.loc[validation_ix]

    datasets = {'init': init_df, 'train': train_df, 'validation': validation_df}

    return datasets


In [12]:
datasets = split_dataset(df=results)

obj = EnduranceEloGradientOptimizer()
output = obj.run(datasets=datasets, num_epochs=50)

{'elo_game_value': 62.70003752940001, 'default_rating': 1573.8907520169992, 'new_agent_alpha': 1.1033114323713464}
Iteration: 1: 0.5612518464995071
Iteration: 2: 0.5572243335547642
Iteration: 3: 0.5522492939508552
Iteration: 4: 0.547513389005317
Iteration: 5: 0.5440645463375042
Iteration: 6: 0.5426155753099048
Iteration: 7: 0.542051972248845
Iteration: 8: 0.5417468537887232
Iteration: 9: 0.5415156357325305
Iteration: 10: 0.5413245667223561
Iteration: 11: 0.5411640604845901
Iteration: 12: 0.5410282643257602
Iteration: 13: 0.5409129370498897
Iteration: 14: 0.5408146723518698
Iteration: 15: 0.5407306975816831
Iteration: 16: 0.540658739443707
Iteration: 17: 0.5405969230220973
Iteration: 18: 0.5405436938916356
Iteration: 19: 0.5404977573126466
Iteration: 20: 0.5404580303142724
Iteration: 21: 0.5404236036190824
Iteration: 22: 0.5403937111598885
Iteration: 23: 0.5403677055083201
Iteration: 24: 0.5403450408405025
Iteration: 25: 0.5403252503585632
Iteration: 26: 0.5403080720935267
Iteration: 27

In [32]:
obj.optimized_parameters

{'elo_game_value': 47.04352098949928,
 'default_rating': 1833.863245818816,
 'new_agent_alpha': 0.7588477980588197}

In [34]:
output

,iteration,elo_game_value,default_rating,new_agent_alpha,pre_train_log_loss,pre_validation_log_loss,elo_game_value_delta,new_elo_game_value,default_rating_delta,new_default_rating,new_agent_alpha_delta,new_new_agent_alpha,new_train_log_loss,new_validation_log_loss
0,1,62.700038,1573.890752,1.103311,0.563477,0.444364,-0.243733,62.456305,16.771046,1590.661798,-0.006719,1.096593,0.561252,0.443721
1,2,62.456305,1590.661798,1.096593,0.561252,0.443721,-0.489758,61.966547,32.495638,1623.157436,-0.013126,1.083467,0.557224,0.442569
2,3,61.966547,1623.157436,1.083467,0.557224,0.442569,-0.735374,61.231173,45.500778,1668.658214,-0.018677,1.064791,0.552249,0.441204
3,4,61.231173,1668.658214,1.064791,0.552249,0.441204,-0.969022,60.262151,52.947335,1721.605549,-0.022840,1.041951,0.547513,0.440042
4,5,60.262151,1721.605549,1.041951,0.547513,0.440042,-1.155946,59.106205,52.523047,1774.128596,-0.024974,1.016976,0.544065,0.439453
5,6,59.106205,1774.128596,1.016976,0.544065,0.439453,-1.070709,58.035496,33.139030,1807.267626,-0.020906,0.996070,0.542616,0.439392
6,7,58.035496,1807.267626,0.996070,0.542616,0.439392,-0.971101,57.064395,16.473104,1823.740730,-0.017611,0.978459,0.542052,0.439418
7,8,57.064395,1823.740730,0.978459,0.542052,0.439418,-0.874993,56.189402,6.186729,1829.927459,-0.015213,0.963247,0.541747,0.439381
8,9,56.189402,1829.927459,0.963247,0.541747,0.439381,-0.789693,55.399709,2.015888,1831.943347,-0.013525,0.949722,0.541516,0.439310
9,10,55.399709,1831.943347,0.949722,0.541516,0.439310,-0.714535,54.685175,0.590616,1832.533963,-0.012227,0.937494,0.541325,0.439237
